In [1]:
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from torch.utils.data import TensorDataset, DataLoader

In [2]:
df = pd.read_csv("../data/raw/Combined_Data.csv")
df = df.dropna()

X = df["statement"].astype(str)
y = df["status"]

In [3]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train).toarray()
X_test = vectorizer.transform(X_test).toarray()

In [6]:
input_size = X_train.shape[1]
num_classes = len(encoder.classes_)

print("Input Size:", input_size)
print("Number of Classes:", num_classes)

Input Size: 50000
Number of Classes: 7


In [7]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64
)

In [9]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

mps


In [10]:
class MentalHealthNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
        nn.Linear(input_size, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.4),

        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(128, len(encoder.classes_)))

    def forward(self,x):
        return self.model(x)

model = MentalHealthNN().to(device)

In [11]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

In [12]:
epochs = 50

for epoch in range(epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = loss_fn(outputs, y_batch)

        loss.backward()

        optimizer.step()

    print(
        f"Epoch {epoch+1} Loss: {loss.item():.4f}"
    )

Epoch 1 Loss: 0.4429
Epoch 2 Loss: 0.5675
Epoch 3 Loss: 0.1633
Epoch 4 Loss: 0.2560
Epoch 5 Loss: 0.1974
Epoch 6 Loss: 0.0574
Epoch 7 Loss: 0.1902
Epoch 8 Loss: 0.2534
Epoch 9 Loss: 0.2044
Epoch 10 Loss: 0.1778
Epoch 11 Loss: 0.3691
Epoch 12 Loss: 0.0841
Epoch 13 Loss: 0.0042
Epoch 14 Loss: 0.0621
Epoch 15 Loss: 0.1206
Epoch 16 Loss: 0.1292
Epoch 17 Loss: 0.2441
Epoch 18 Loss: 0.1711
Epoch 19 Loss: 0.0709
Epoch 20 Loss: 0.1623
Epoch 21 Loss: 0.0629
Epoch 22 Loss: 0.1051
Epoch 23 Loss: 0.0149
Epoch 24 Loss: 0.2227
Epoch 25 Loss: 0.0530
Epoch 26 Loss: 0.3589
Epoch 27 Loss: 0.0093
Epoch 28 Loss: 0.0835
Epoch 29 Loss: 0.0897
Epoch 30 Loss: 0.1334
Epoch 31 Loss: 0.3854
Epoch 32 Loss: 0.0318
Epoch 33 Loss: 0.0474
Epoch 34 Loss: 0.0841
Epoch 35 Loss: 0.0166
Epoch 36 Loss: 0.2818
Epoch 37 Loss: 0.0167
Epoch 38 Loss: 0.0398
Epoch 39 Loss: 0.0552
Epoch 40 Loss: 0.0358
Epoch 41 Loss: 0.2142
Epoch 42 Loss: 0.2082
Epoch 43 Loss: 0.0347
Epoch 44 Loss: 0.1735
Epoch 45 Loss: 0.0262
Epoch 46 Loss: 0.34

In [13]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

model.eval()

predictions = []

with torch.no_grad():

    for X_batch, _ in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        preds = outputs.argmax(1)

        predictions.extend(preds.cpu().numpy())

accuracy = accuracy_score(y_test, predictions)

precision = precision_score(
    y_test,
    predictions,
    average="weighted"
)

recall = recall_score(
    y_test,
    predictions,
    average="weighted"
)

f1 = f1_score(
    y_test,
    predictions,
    average="weighted"
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.7821
Precision: 0.7812
Recall   : 0.7821
F1 Score : 0.7808
